# Multi-Scale SR — Google Colab GPU Training

Pull the `multiscale_sr` code from **your fork** (`rajveer43/CMS`, `main`), read the CMS jet parquet data from **Google Drive** (or an uploaded folder), train a chosen scale on the **full dataset** on the Colab GPU with a **multi-seed sweep**, evaluate tagging efficiency (+ SEMD), and save everything to **Google Drive** so results survive the session ending.

## Run order

Train **64×64 first, then 32×32, then 16×16** — set `SCALE` in Cell 6 and *Run all* once per scale, in that order. Each scale is a separate Run-All (see Cell 10); this is deliberate, not an oversight — it keeps each scale's multi-seed sweep resumable on its own and avoids losing later scales to a single long Colab session timing out.

## Before you Run-All (prerequisites)

- **(P1) Push your latest code to `rajveer43/CMS` first.** This notebook pulls code from git — any uncommitted local work will *not* be here. Commit + push the branch you want, then set `REPO_BRANCH` in **Cell 2** to match (defaults to `main`).
- **(P2) Put the `*.parquet` jet files somewhere Colab can read.** Easiest is Google Drive. Set `DATA_DIR` in **Cell 4** to that folder.
- **(P3) Runtime settings:** *Runtime → Change runtime type → Hardware accelerator = GPU* (T4 is fine). Colab needs internet for `git clone` + `pip` (on by default).
- **(P4) Optional W&B:** add `WANDB_API_KEY` via the **🔑 Secrets** panel (left sidebar) with notebook access enabled, or paste it in **Cell 5**. No key → training runs with `--no-wandb`.

## How to run (repeat for 64 → 32 → 16)
1. Set `DATA_DIR` in **Cell 4**.
2. Set `SCALE` in **Cell 6** — start with `64`.
3. *Runtime → Run all*.
4. Results are saved to Drive as each seed finishes (Cell 7) and reviewed in Cell 9. Nothing to download manually.
5. Set `SCALE = 32` in Cell 6, *Run all* again. Then `SCALE = 16`, *Run all* a third time.


## Cell 1 — Environment check (GPU is required)

In [ ]:
import subprocess, sys
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU. Enable it: Runtime -> Change runtime type -> Hardware accelerator = GPU (T4), "
        "then Runtime -> Run all again."
    )
print("device:", torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

torch: 2.11.0+cu128
cuda available: True
device: NVIDIA A100-SXM4-80GB
Thu Aug 13 15:33:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             51W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        | 

## Cell 2 — Get the code (clone your fork at a pinned branch)
Clones `rajveer43/CMS` (your fork) — not upstream `ML4SCI/CMS`. Re-running is safe: the clone dir is removed first. Set `REPO_BRANCH` to whatever you pushed in (P1); defaults to `main`. `CODE_DIR` points at `multiscale_sr/` nested under `E2E/E2E_Super_Resolution_Rajveer_Rathod/`, matching this repo's actual layout.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/rajveer43/CMS.git"
REPO_BRANCH = "main"
CLONE_DIR = Path("/content/repo")
CODE_DIR = CLONE_DIR / "E2E" / "E2E_Super_Resolution_Rajveer_Rathod" / "multiscale_sr"

# Always leave the repo before deleting it
os.chdir("/content")

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        REPO_BRANCH,
        REPO_URL,
        str(CLONE_DIR),
    ],
    check=True,
)

os.chdir(CODE_DIR)
print("Current directory:", os.getcwd())

Current directory: /content/repo/multiscale_sr


## Cell 3 — Dependencies (install only what's missing)
Colab ships torch/numpy/pyarrow/sklearn/matplotlib/pyyaml. We only add `wandb` and `python-dotenv`; torch is **not** touched.

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
        print(f"ok: {pkg}")
    except ImportError:
        print(f"installing: {pkg}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, imp in [("wandb", "wandb"), ("python-dotenv", "dotenv"), ("psutil", "psutil")]:
    ensure(pkg, imp)

import torch, numpy, pyarrow, sklearn
print("torch", torch.__version__, "| numpy", numpy.__version__,
      "| pyarrow", pyarrow.__version__, "| sklearn", sklearn.__version__)

ok: wandb
ok: python-dotenv
torch 2.11.0+cu128 | numpy 2.0.2 | pyarrow 18.1.0 | sklearn 1.6.1


## Cell 4 — Locate the data (Google Drive)
Mounts your Google Drive and points `DATA_DIR` at the folder holding the `*.parquet` files. **Edit `DATA_DIR`** to your actual path, then run. If you'd rather upload the files directly instead of using Drive, skip the mount and set `DATA_DIR` to wherever you put them (e.g. `/content/datasets`).

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

# --- Mount Google Drive (comment out if you uploaded data to /content instead) ---
from google.colab import drive
drive.mount("/content/drive")

# ================== EDIT THIS ONE LINE ==================
DATA_DIR = Path("/content/drive/MyDrive/GSoC_2026/DATASETS/QUARK_GLUON")   # <- folder that CONTAINS the *.parquet files
# =======================================================

files = sorted(DATA_DIR.glob('*.parquet'))
if not files:
    raise SystemExit(
        f"No *.parquet found in {DATA_DIR}. Fix DATA_DIR above to the folder that holds the "
        f"jet parquet files (list what is there with: !ls \"{DATA_DIR}\")."
    )

total = 0
print("DATA_DIR:", DATA_DIR)
for f in files:
    n = pq.ParquetFile(f).metadata.num_rows
    total += n
    print(f"  {f.name}: {n:,} rows")
print(f"TOTAL: {total:,} jets across {len(files)} file(s)")
print("(code splits by file: first files -> train, last -> val)")

Mounted at /content/drive
DATA_DIR: /content/drive/MyDrive/GSoC_2026/DATASETS/QUARK_GLUON
  QCDToGGQQ_IMGjet_RH1all_jet0_run0_n36272_LR.parquet: 36,272 rows
  QCDToGGQQ_IMGjet_RH1all_jet0_run1_n47540_LR.parquet: 47,540 rows
  QCDToGGQQ_IMGjet_RH1all_jet0_run2_n55494_LR.parquet: 55,494 rows
TOTAL: 139,306 jets across 3 file(s)
(code splits by file: first files -> train, last -> val)


## Cell 5 — W&B (optional)
Reads `WANDB_API_KEY` from the Colab **Secrets** panel (🔑, left sidebar) if present, else falls back to `--no-wandb`. You can also paste a key into the marked line. Never commit a key.

In [ ]:
import os

USE_WANDB = False

# Option A: Colab Secrets panel (recommended). Add a secret named WANDB_API_KEY and
# toggle notebook access on.
try:
    from google.colab import userdata
    _key = userdata.get("WANDB_API_KEY")
    if _key:
        os.environ["WANDB_API_KEY"] = _key
        USE_WANDB = True
        print("W&B: key found in Colab Secrets -> logging ENABLED")
except Exception as e:
    print("W&B: no Colab secret ->", type(e).__name__)

# Option B: paste a key here instead (leave empty to skip).
if not USE_WANDB:
    _PASTED_KEY = ""   # <- optionally paste your wandb key
    if _PASTED_KEY:
        os.environ["WANDB_API_KEY"] = _PASTED_KEY
        USE_WANDB = True
        print("W&B: using pasted key -> logging ENABLED")

if not USE_WANDB:
    print("W&B: DISABLED (training will use --no-wandb).")

W&B: key found in Colab Secrets -> logging ENABLED


## Cell 6 — Parameters (edit these)
Full dataset per epoch (no batch cap). On a T4, an ~84k-sample epoch is a few minutes, so a 20–30 epoch run fits comfortably. Results are written under `/content/experiments` and also copied to Drive at the end so they survive the session ending.

In [ ]:
SCALE            = 64                 # 64 -> 32 -> 16, IN THAT ORDER (one Run-All per scale; see Cell 10)
EPOCHS           = 40                 # (change this for different epoch training)
# RUN_NAME is now dynamically generated in Cell 7 for multi-seed runs.
EXPERIMENTS_ROOT = "/content/experiments"
SEED             = 42                 # This is overridden in Cell 7 for multi-seed runs.
RESUME           = None               # e.g. '.../checkpoints/latest.pt' to continue

CONFIG = f"configs/scale_{SCALE}.yaml"
assert Path(CONFIG).exists(), f"missing {CONFIG} in {os.getcwd()} — check the clone (Cell 2)"
print(f"Base parameters for multi-seed runs (used in run name generation):")
print(f"scale={SCALE}  epochs={EPOCHS}  wandb={USE_WANDB}")
print(f"config={CONFIG}  data={DATA_DIR}  experiments_root={EXPERIMENTS_ROOT}")

Base parameters for multi-seed runs (used in run name generation):
scale=64  epochs=40  wandb=True
config=configs/scale_64.yaml  data=/content/drive/MyDrive/GSoC_2026/DATASETS/QUARK_GLUON  experiments_root=/content/experiments


In [ ]:
import shutil
from pathlib import Path
LOCAL_DATA = Path("/content/datasets")
LOCAL_DATA.mkdir(exist_ok=True)
for f in DATA_DIR.glob("*.parquet"):
    dst = LOCAL_DATA / f.name
    if not dst.exists():
        print("copying", f.name)
        # copy2 preserves mtime. The sweep's dataset fingerprint hashes
        # (name, size, mtime_ns) -- see colab_sweep.py:dataset_manifest -- so a
        # plain shutil.copy would mint a NEW fingerprint on every fresh VM,
        # hiding previously completed seeds and forcing a full cache rebuild.
        shutil.copy2(f, dst)
DATA_DIR = LOCAL_DATA   # train from local disk, not Drive


copying QCDToGGQQ_IMGjet_RH1all_jet0_run0_n36272_LR.parquet
copying QCDToGGQQ_IMGjet_RH1all_jet0_run1_n47540_LR.parquet
copying QCDToGGQQ_IMGjet_RH1all_jet0_run2_n55494_LR.parquet


## Cell 6b — Preflight: disk and RAM capacity for the decode cache

`--cache` decodes **all** HR jets into a float32 memmap before the first
training step, and the train and held-out splits are cached **separately**
(`factory.py`), so together they total roughly the whole dataset. On top of
that, the cell above keeps a full local copy of the raw parquet.

A container that runs out of RAM or disk mid-build is killed by the host: the
Colab runtime simply disconnects, with no Python traceback. This cell fails
loudly *before* that happens and tells you which resource is short.


In [ ]:
# PREFLIGHT_CAPACITY_CHECK
import shutil
import pyarrow.parquet as pq
from pathlib import Path

_files = sorted(Path(DATA_DIR).glob("*.parquet"))
if not _files:
    raise SystemExit(f"No parquet in {DATA_DIR} - run the data cells above first.")

# Probe HR shape from one row; cache is float32 (n, C, H, W).
_probe = next(pq.ParquetFile(_files[0]).iter_batches(batch_size=1, columns=["X_jets"]))
import numpy as _np
_arr = _np.array(_probe.column(0).to_pylist()[0], dtype=_np.float32)
_c, _h, _w = (_arr.shape if _arr.ndim == 3 else (_arr.shape[-1], *_arr.shape[:2]))
_n = sum(pq.ParquetFile(f).metadata.num_rows for f in _files)

_cache_bytes = _n * _c * _h * _w * 4          # train + heldout ~= all rows once
_raw_bytes = sum(f.stat().st_size for f in _files)
_need = _cache_bytes + _raw_bytes
_free = shutil.disk_usage("/content").free

print(f"jets={_n:,}  HR=({_c},{_h},{_w})  decode cache ~{_cache_bytes/1e9:.1f} GB")
print(f"raw parquet ~{_raw_bytes/1e9:.1f} GB   ->  need ~{_need/1e9:.1f} GB")
print(f"/content free: {_free/1e9:.1f} GB")

try:
    import psutil
    _ram = psutil.virtual_memory().total
    print(f"RAM total: {_ram/1e9:.1f} GB")
    # Dirty memmap pages are flushed every 512MB during the build, so peak RSS
    # is bounded well below cache size; a small floor still catches the
    # standard ~12.7GB runtime being used for a very large dataset.
    if _ram < 8e9:
        print("WARNING: low RAM runtime. Prefer Runtime -> Change runtime type -> High-RAM.")
except ImportError:
    print("psutil missing - RAM not checked")

if _free < _need * 1.10:                       # 10% headroom
    raise SystemExit(
        f"Not enough disk on /content: need ~{_need/1e9:.1f} GB (+10% headroom), "
        f"free {_free/1e9:.1f} GB. The cache build would be killed by the host "
        f"mid-write and disconnect the runtime. Free space, use a larger runtime, "
        f"or point --cache-dir at a volume with room."
    )
print("OK: capacity sufficient for the decode cache.")


## Cell 7 — Multi-seed sweep: train + evaluate + SAVE TO DRIVE, one seed at a time

Each seed is trained, evaluated, and **copied to Drive before the next seed starts**, so a Colab disconnect costs at most the run in flight. Already-saved seeds are skipped, so re-running this cell resumes a broken sweep.

Two things are deliberately held fixed across all seeds: `EVAL_SEED` and a single frozen HR tagger. Only the training seed varies, so the spread you measure is the model's, not the evaluation's. Watch that **HR AUC is identical for every seed** — if it moves, a seed has leaked into the measurement.

In [ ]:
# import multiscale_sr.utils.env as _env
# _orig = _env.resolve_env
# def _patched():
#     c = _orig()
#     return c.__class__(**{**c.__dict__, "num_workers": 0,
#                           "persistent_workers": False, "prefetch_factor": None})
# _env.resolve_env = _patched
# print("forced num_workers=0")


In [ ]:
import json, subprocess, sys
from pathlib import Path

SEEDS_TO_RUN = [123, 456, 789, 999]
RUN_VARIANT = "learnable_lr_skip_v1"
DRIVE_RUNS_ROOT = Path("/content/drive/MyDrive/multiscale_sr_runs")
EVAL_SEED = 0
SEMD_TOPK = 128
SEMD_OMEGA_R = 1.0
SEMD_MAX_SAMPLES = 1000
SEMD_CHUNK = 1
CHECKPOINT_SECONDS = 300
# Resume is automatic and specific to each seed. Never broadcast one checkpoint.
if RESUME:
    raise ValueError("Set RESUME=None: the sweep locates each seed's verified recovery checkpoint.")
result_file = Path("/content/current_multiscale_sweep.json")
cmd = [
    sys.executable, "-u", "colab_sweep.py",
    "--config", CONFIG, "--data-dir", str(DATA_DIR),
    "--drive-root", str(DRIVE_RUNS_ROOT), "--local-root", str(EXPERIMENTS_ROOT),
    "--scale", str(SCALE), "--epochs", str(EPOCHS),
    "--variant", RUN_VARIANT, "--seeds", *map(str, SEEDS_TO_RUN),
    "--eval-seed", str(EVAL_SEED), "--semd-topk", str(SEMD_TOPK),
    "--semd-omega-R", str(SEMD_OMEGA_R), "--semd-max-samples", str(SEMD_MAX_SAMPLES),
    "--semd-chunk", str(SEMD_CHUNK), "--checkpoint-seconds", str(CHECKPOINT_SECONDS),
    "--result-file", str(result_file),
]
if not USE_WANDB:
    cmd.append("--no-wandb")
# Run in a fresh Python process so re-cloning cannot leave stale model imports.
result_file.unlink(missing_ok=True)
sweep_process = subprocess.run(cmd)
if not result_file.exists():
    raise RuntimeError("Sweep failed before initialization; inspect the traceback above.")
sweep_summary = json.loads(result_file.read_text())
SWEEP_ROOT = Path(sweep_summary["sweep_root"])
TRAINING_FINGERPRINT = sweep_summary["training_fingerprint"]
EVAL_ID = sweep_summary["eval_id"]
saved_to_drive = {int(s): Path(r["path"]) for s, r in sweep_summary["results"].items()
                  if r["status"] == "complete"}
print("Summary:", SWEEP_ROOT / f"summary_{EVAL_ID}.json")
if sweep_process.returncode:
    print("Sweep incomplete. Read the seed logs; re-running this cell resumes verified checkpoints.")


## Cell 8 — Evaluate tagging efficiency on the trained checkpoint

In [ ]:
# This cell is now redundant. Its functionality has been moved into Cell 7 for multi-seed execution.

## Cell 9 — Review the sweep (already saved to Drive)
Each seed was copied to Drive by Cell 7 the moment it finished, so nothing is copied here. This cell builds the per-seed comparison table and checks that HR AUC stayed fixed across seeds.

In [ ]:
import json
import statistics as st

rows = []
for seed in SEEDS_TO_RUN:
    status = sweep_summary["results"].get(str(seed), {"status": "pending"})
    if status["status"] != "complete":
        print(f"seed {seed}: {status}")
        continue
    run_dir = Path(status["path"])
    jf = run_dir / "evaluations" / EVAL_ID / "classification_eval.json"
    p = json.loads(jf.read_text())["primary_fixed_hr_tagger"]
    rows.append((seed, p["auc"]["hr"], p["auc"]["lr"], p["auc"]["sr"],
                 p["tagging_efficiency_sr_over_hr"]))
print(" seed     HR AUC    LR AUC    SR AUC    efficiency")
for seed, hr, lr, sr, eff in rows:
    print(f"{seed:5d}   {hr:.4f}    {lr:.4f}    {sr:.4f}    {eff:.4f}")
if len(rows) > 1:
    print(f"SR AUC: {st.mean(r[3] for r in rows):.4f} +/- {st.stdev(r[3] for r in rows):.4f}")
    print(f"HR AUC spread: {max(r[1] for r in rows) - min(r[1] for r in rows):.3g}")
# No automatic full-Drive ZIP: large unrelated experiments can exhaust local disk.


## Cell 9a — Evaluation recovery
Cell 7 automatically retries failed evaluations using the saved trained checkpoint. Legacy results are excluded from this sweep.


In [ ]:
# Evaluation retries are automatic in Cell 7.
# Legacy checkpoint retro-evaluation must be run explicitly as a separate analysis.


## Cell 9b — Does SEMD see what the pixel-wise metrics cannot?

This is the decision point. Across the four seeds, tagging efficiency swings ~10%
while every logged physics metric stays flat to <1% — because all of them
(`energy_response`, `peak_ratio`, `nonzero_ratio`) are **invariant under pixel
permutation** and therefore blind to *where* energy sits. SEMD is not.

The cell below correlates **every** metric against tagging efficiency, SEMD
included and with no special treatment, and prints a verdict.

**Read the caveat it prints.** With n=4 the smallest attainable permutation
p-value is 0.083, so nothing here can reach significance — it ranks hypotheses,
it does not confirm one. If SEMD does *not* win, that is the finding: report it,
and do not sweep `topk`/`omega_R` hunting for a setting that reproduces the
ranking you already know.

In [ ]:
# Build a report from exactly the completed seeds in this sweep/evaluation.
import shutil, tempfile
report_dir = SWEEP_ROOT / "reports" / EVAL_ID
report_dir.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory() as tmp:
    for seed, run in saved_to_drive.items():
        target = Path(tmp) / f"seed_{seed}" / "figures" / "classification"
        target.mkdir(parents=True)
        shutil.copyfile(run / "evaluations" / EVAL_ID / "classification_eval.json",
                        target / "classification_eval.json")
    if saved_to_drive:
        subprocess.run([sys.executable, "-u", "semd_correlation.py",
            "--eval-dir", tmp, "--scale", str(SCALE),
            "--out", str(report_dir / "semd_correlation.md"),
            "--out-json", str(report_dir / "semd_correlation.json")], check=True)
    else:
        print("No completed seeds to report.")


## Cell 10 — After the sweep

Results are under `multiscale_sr_runs/<variant>_<scale>x_<fingerprint>/seed_<seed>/attempt_<id>/`.
Verified rotating checkpoints are saved every five minutes (at batch boundaries) and before validation. Re-run Cell 7 to resume each interrupted seed. This supports the cached parquet loader with zero workers; a hard kill loses work since the last successfully persisted checkpoint.

Read `summary_<eval_id>.json`, per-attempt `train-*.log` / `eval-*.log`, and `resources-*.jsonl` for failures. Reports include only completed seeds from this sweep. See the [Colab reliability guide](../docs/COLAB_RELIABILITY.md) for the smoke-test and recovery procedure.

For another scale, set `SCALE` and Run all: 64 → 32 → 16. Full-duration Colab testing is still required before interpreting local tests as proof of runtime stability.
